# Analise de Dados - Camada Gold
Este notebook responde as perguntas de negocio e gera a camada Gold agregada (VIEW e TABELA), com JOIN + GROUP BY.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import banco
import warnings
warnings.filterwarnings('ignore')

conexao = banco.conectar()
sns.set_theme(style='whitegrid')

### Parte 1: Camada Silver - AnÃ¡lises Diretas

In [ ]:
# 1. Os 5 Ã³rgÃ£os com maior custo total?
q1 = "SELECT nome_orgao_superior, SUM(valor_total) as custo FROM silver_viagem GROUP BY nome_orgao_superior ORDER BY custo DESC LIMIT 5"
df1 = pd.read_sql(q1, conexao)
display(df1)
plt.figure(figsize=(10,4))
sns.barplot(data=df1, y='nome_orgao_superior', x='custo', palette='viridis')
plt.title('Top 5 Ã“rgÃ£os com Maior Custo Total')
plt.xlabel('Custo Total (R$)')
plt.ylabel('Ã“rgÃ£o Superior')
plt.show()

# 2. Os 3 destinos com maior custo mÃ©dio por viagem?
q2 = "SELECT destinos, AVG(valor_total) as custo_medio FROM silver_viagem GROUP BY destinos ORDER BY custo_medio DESC LIMIT 3"
df2 = pd.read_sql(q2, conexao)
display(df2)
plt.figure(figsize=(10,4))
sns.barplot(data=df2, y='destinos', x='custo_medio', palette='magma')
plt.title('Top 3 Destinos com Maior Custo MÃ©dio')
plt.xlabel('Custo MÃ©dio (R$)')
plt.ylabel('Destinos')
plt.show()

# 3. A viagem de maior duraÃ§Ã£o e seu custo total?
q3 = "SELECT id_viagem, duracao_dias, valor_total FROM silver_viagem ORDER BY duracao_dias DESC LIMIT 1"
df3 = pd.read_sql(q3, conexao)
display(df3)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['DuraÃ§Ã£o'], df3['duracao_dias'], color='steelblue')
axes[0].set_title('DuraÃ§Ã£o (dias)')
axes[0].set_xlabel('Viagem ' + str(df3['id_viagem'].iloc[0]))
axes[0].set_ylabel('Dias')
axes[1].bar(['Custo Total'], df3['valor_total'], color='darkorange')
axes[1].set_title('Custo Total (R$)')
axes[1].set_xlabel('Viagem ' + str(df3['id_viagem'].iloc[0]))
axes[1].set_ylabel('Valor (R$)')
plt.suptitle('Viagem de Maior DuraÃ§Ã£o')
plt.tight_layout()
plt.show()


### Parte 2: ConstruÃ§Ã£o da Camada Gold Agregada
ConstruÃ­mos a camada Gold como uma **VIEW** e como uma **TABELA fÃ­sica**, agregando `silver_viagem` com `silver_pagamento` e `silver_trecho` por meio de JOIN + GROUP BY.

**ObservaÃ§Ã£o de design:** pagamento e trecho sÃ£o prÃ©-agregados (uma linha por `id_viagem`) *antes* do JOIN com viagem. Se juntÃ¡ssemos as 3 tabelas diretamente (viagem 1:N pagamento 1:N trecho), viagens com mais de um pagamento **e** mais de um trecho gerariam um produto cartesiano (ex.: 2 pagamentos Ã— 3 trechos = 6 linhas em vez de 5), inflando somas e contagens. Por isso, as perguntas 4 a 7 (que dependem de `tipo_pagamento`, `meio_transporte` e `destino_uf`, atributos de nÃ­vel pagamento/trecho) sÃ£o respondidas consultando `silver_pagamento` e `silver_trecho` diretamente â€” sem esse risco de duplicaÃ§Ã£o.


In [ ]:
# Construindo a camada Gold agregada (JOIN + GROUP BY) no banco de dados.
# pagamento e trecho sao pre-agregados por id_viagem (GROUP BY) ANTES do JOIN
# com viagem, para nao duplicar linhas (fan-out) quando uma viagem tem mais
# de um pagamento e mais de um trecho ao mesmo tempo.

sql_gold_view = """
CREATE OR REPLACE VIEW gold_resumo_viagens AS
SELECT
    v.id_viagem,
    v.nome_orgao_superior,
    v.valor_total                   AS custo_viagem,
    v.duracao_dias,
    COALESCE(pag.total_pago, 0)     AS total_pago,
    COALESCE(pag.qtd_pagamentos, 0) AS qtd_pagamentos,
    COALESCE(tre.qtd_trechos, 0)    AS qtd_trechos
FROM silver_viagem v
LEFT JOIN (
    SELECT id_viagem, SUM(valor) AS total_pago, COUNT(*) AS qtd_pagamentos
    FROM silver_pagamento
    GROUP BY id_viagem
) pag ON pag.id_viagem = v.id_viagem
LEFT JOIN (
    SELECT id_viagem, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    GROUP BY id_viagem
) tre ON tre.id_viagem = v.id_viagem;
"""
banco.executar(conexao, sql_gold_view)
print("VIEW gold_resumo_viagens criada com sucesso!")

# Materializa a mesma agregaÃ§Ã£o como TABELA fÃ­sica (requisito: tabela e view)
banco.executar(conexao, "DROP TABLE IF EXISTS gold_resumo_viagens_tab;")
banco.executar(conexao, "CREATE TABLE gold_resumo_viagens_tab AS SELECT * FROM gold_resumo_viagens;")
print("TABELA gold_resumo_viagens_tab criada com sucesso!")

display(pd.read_sql("SELECT * FROM gold_resumo_viagens_tab LIMIT 5", conexao))


### Parte 3: Camada Gold - Analises Agregadas

In [ ]:
# 4. Qual o tipo de pagamento com maior valor mÃ©dio?
# Consulta direta em silver_pagamento (juntar com trecho infla os valores - ver Parte 2)
q4 = "SELECT tipo_pagamento, AVG(valor) as valor_medio FROM silver_pagamento GROUP BY tipo_pagamento ORDER BY valor_medio DESC LIMIT 5"
df4 = pd.read_sql(q4, conexao)
display(df4)

# 5. Qual o meio de transporte mais usado nos trechos?
q5 = "SELECT meio_transporte, COUNT(*) as qtd FROM silver_trecho GROUP BY meio_transporte ORDER BY qtd DESC"
df5 = pd.read_sql(q5, conexao)
display(df5)
plt.figure(figsize=(10,4))
sns.barplot(data=df5, x='meio_transporte', y='qtd', palette='coolwarm')
plt.title('Meios de Transporte Mais Utilizados')
plt.xlabel('Meio de Transporte')
plt.ylabel('Quantidade de Trechos')
plt.show()

# 6. Qual UF de destino aparece em mais trechos?
q6 = "SELECT destino_uf, COUNT(*) as qtd FROM silver_trecho GROUP BY destino_uf ORDER BY qtd DESC LIMIT 5"
df6 = pd.read_sql(q6, conexao)
display(df6)
plt.figure(figsize=(10,4))
sns.barplot(data=df6, x='destino_uf', y='qtd', palette='crest')
plt.title('Top 5 UFs de Destino')
plt.xlabel('UF (Unidade Federativa)')
plt.ylabel('Quantidade de Trechos')
plt.show()

# 7. Qual Ã³rgÃ£o pagou mais no total?
q7 = "SELECT nome_orgao_pagador, SUM(valor) as total_pago FROM silver_pagamento GROUP BY nome_orgao_pagador ORDER BY total_pago DESC LIMIT 5"
df7 = pd.read_sql(q7, conexao)
display(df7)
plt.figure(figsize=(10,4))
sns.barplot(data=df7, y='nome_orgao_pagador', x='total_pago', palette='rocket')
plt.title('Top 5 Ã“rgÃ£os Pagadores pelo Total Pago')
plt.xlabel('Valor Total Pago (R$)')
plt.ylabel('Ã“rgÃ£o Pagador')
plt.show()
